# Kiểm chứng câu trả lời QA bằng NLI: chạy toàn bộ pipeline

**Chuẩn bị (một lần):**
1. Thư mục `nlp_doan/` (chứa `doan.zip`) nằm trên Drive của một người và được **chia sẻ quyền Editor** cho người kia.
   Người được chia sẻ bấm chuột phải vào thư mục → **Organize → Add shortcut** → chọn **My Drive**, để cả hai
   cùng thấy đường dẫn `MyDrive/nlp_doan/`.
2. Chọn **Runtime → Change runtime type → T4 GPU**.
3. Ở cell Cấu hình, mỗi người đặt `ROLE` của mình rồi **Runtime → Run all**.

**Chạy song song (N = 11873, toàn bộ SQuAD 2.0 dev):**

| ROLE | Việc | Thời gian ước tính trên T4 (chưa đo) |
|---|---|---|
| `"A"` | reader, rồi nli-xs và nli-base | reader ~10–15 phút, mỗi verifier ~30 phút |
| `"B"` | đợi `preds.jsonl` của A xuất hiện trên Drive, rồi nli-large | ~2.5–4 giờ |
| `"all"` | một người chạy tất cả (bước nào xong rồi thì tự bỏ qua) | ~4–5 giờ |

`FINETUNE = True` thêm bước 4b (tinh chỉnh verifier, ~30–45 phút) cho ROLE `"all"` hoặc `"B"`. Nếu 3 verifier đã chấm xong,
đặt `ROLE = "all"` rồi Run all: chỉ bước 4b và phần tổng hợp chạy.

Nếu Colab bị ngắt, chỉ cần Run all lại: các bước đã xong tự bỏ qua, verifier đang chạy dở tiếp tục từ chỗ dừng
(mất tối đa 200 câu).

**Hết quota GPU giữa chừng?** Chạy tiếp bằng tài khoản khác: chia sẻ `nlp_doan/` (Editor) cho tài khoản đó, thêm shortcut
vào My Drive, đặt **cùng ROLE** với phiên bị ngắt rồi Run all; verifier tiếp tục từ chỗ dừng. Chỉ chạy **một phiên cho mỗi
verifier** tại một thời điểm (đóng hẳn phiên cũ trước). Phần tổng hợp chỉ chạy khi **đủ cả 3 verifier**, nên ai chạy xong sau thì Run all lại để có kết quả.

## 1. Cấu hình

In [ ]:
ROLE = "all"                                     # "A", "B" hoặc "all" (xem bảng ở trên)
DRIVE_DIR = "/content/drive/MyDrive/nlp_doan"   # thư mục chứa doan.zip (shortcut nếu được chia sẻ)
N = 11873                                        # số câu hỏi SQuAD 2.0 dev (11873 = toàn bộ; 2000 = bản thử)
RUN = f"squad{N // 1000}k" if N >= 1000 else f"squad{N}"
ALL_VERIFIERS = ["nli-xs", "nli-base", "nli-large"]
VERIFIERS = {"A": ["nli-xs", "nli-base"], "B": ["nli-large"], "all": ALL_VERIFIERS}[ROLE]
FINETUNE = True                                  # bước 4b: tinh chỉnh verifier trên SQuAD 2.0 train rồi chấm như verifier mới
# Mô hình tinh chỉnh cần có: tên -> tham số cho finetune_verifier.py (đã có trên Drive thì bỏ qua bước huấn luyện)
FT_MODELS = {
    "nli-base-qa":  "--base nli-base",                                   # đã chạy
    "nli-base-qa2": "--base nli-base --n_swap 20000",                    # + mẫu âm đổi thực thể (~1 giờ)
    "nli-large-qa": "--base nli-large --batch_size 8 --grad_accum 2",    # bản large (~3–4 giờ)
    # Thí nghiệm mới (xếp theo độ ưu tiên; xoá dòng nào không kịp chạy)
    "nli-base-qa3": "--base nli-base --n_pairs 10000",                   # + 10k cặp câu gần giống tự nhiên
    "nli-base-qa3-s14": "--base nli-base --n_pairs 10000 --seed 14",     # seed khác để đo độ dao động
    "nli-base-qa-s14": "--base nli-base --seed 14",
    "nli-base-qa3-s15": "--base nli-base --n_pairs 10000 --seed 15",
    "nli-base-qa-s15": "--base nli-base --seed 15",
}
# Đánh giá ngoài miền (bước 4c): tên run -> (bộ dữ liệu, số câu)
OOD_SETS = {
    "addsent": ("addsent", 2560),                       # SQuAD 1.1 + câu gây nhiễu đối kháng (Jia & Liang 2017)
    "addsent_orig": ("addsent_orig", 1000),             # 1000 câu gốc tương ứng, để so sánh
    "nq2k": ("mrqa:NaturalQuestionsShort", 2000),       # câu hỏi thật từ Google Search
    "newsqa2k": ("mrqa:NewsQA", 2000),                  # tin tức CNN
}
OOD_VERIFIERS = ["nli-base", "nli-large", "nli-base-qa", "nli-large-qa"]
if FINETUNE:
    ALL_VERIFIERS = ALL_VERIFIERS + list(FT_MODELS)
USE_LLM = False                                  # giả thuyết LLM không giúp ở bản 2k -> tắt cho bản 11k
LLM = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"ROLE={ROLE}  RUN={RUN}  verifiers={VERIFIERS}")

## 2. Kết nối Drive, kiểm tra GPU, cài đặt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
assert os.path.exists(f"{DRIVE_DIR}/doan.zip"), f"không thấy {DRIVE_DIR}/doan.zip (đã thêm shortcut chưa?)"
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
!rm -rf /content/doan && unzip -q -o "{DRIVE_DIR}/doan.zip" -d /content
# runs/ và data/ nằm trên Drive để giữ lại kết quả khi Colab ngắt
!ln -sfn "{DRIVE_DIR}/runs" /content/doan/runs
!mkdir -p "{DRIVE_DIR}/data" && ln -sfn "{DRIVE_DIR}/data" /content/doan/data
%cd /content/doan
!pip install -q sentencepiece protobuf
!python -m pytest -q tests

## 3. Bước 1: reader và top-5 ứng viên (ROLE A hoặc all)

ROLE B không chạy reader mà đợi file của A (Drive giữa hai tài khoản có thể trễ vài phút).
Nếu đợi quá 60 phút, chạy lại cell này.

In [ ]:
import json, time
preds = f"runs/{RUN}/preds.jsonl"


def n_lines(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


if ROLE in ("A", "all"):
    # Nếu preds.jsonl đã có thì reader không chạy lại; với USE_LLM chỉ bổ sung giả thuyết llm
    llm_flag = f"--llm {LLM}" if USE_LLM else ""
    !python scripts/run_reader.py --n {N} --out runs/{RUN} --batch_size 32 {llm_flag}
else:
    for minute in range(60):
        if os.path.exists(preds) and n_lines(preds) == N:
            break
        print(f"đợi {preds} từ ROLE A... ({minute} phút)", flush=True)
        time.sleep(60)
assert os.path.exists(preds) and n_lines(preds) == N, f"{preds} chưa có đủ {N} câu"
print("preds.jsonl:", n_lines(preds), "câu")

## 4. Bước 2: NLI verifier (mỗi ROLE chạy phần của mình)

In [ ]:
hyps = "concat,template,llm" if USE_LLM else "concat,template"
for v in VERIFIERS:
    bs = 16 if v == "nli-large" else 64
    !python scripts/run_verify.py --run runs/{RUN} --verifier {v} --hyps {hyps} --batch_size {bs} --group 200

## 4b. Tinh chỉnh verifier trên SQuAD 2.0 train (FINETUNE, chạy ở ROLE all hoặc B)

Mỗi mô hình trong `FT_MODELS`: tạo cặp QA→NLI từ SQuAD 2.0 **train** (đáp án đúng → entail; plausible answer của câu
không đáp án, đáp án của câu hỏi khác, và với `--n_swap` câu hỏi bị đổi thực thể → không kéo theo), tinh chỉnh 1 epoch,
rồi chấm tập `RUN` với giả thuyết template. Mô hình đã có trên Drive thì bỏ qua huấn luyện; chấm dở thì tiếp tục từ chỗ dừng.
Hết quota giữa chừng: Run all lại (cùng tài khoản hoặc tài khoản khác), các mô hình đã xong tự bỏ qua.
Ước tính trên T4 (chưa đo): bản base ~45–60 phút, bản large ~3–4 giờ.

In [ ]:
if FINETUNE and ROLE in ("B", "all"):
    for name, ft_args in FT_MODELS.items():
        ft_dir = f"runs/models/{name}"
        print(f"===== {name}")
        if not os.path.exists(f"{ft_dir}/config.json"):
            !python scripts/finetune_verifier.py --out {ft_dir} {ft_args}
        if not os.path.exists(f"{ft_dir}/config.json"):
            print(f"!!! {name}: huấn luyện thất bại (xem lỗi ở trên), bỏ qua bước chấm điểm")
            continue
        !cat {ft_dir}/train_info.json
        bs = 16 if "large" in name else 64
        !python scripts/run_verify.py --run runs/{RUN} --verifier {ft_dir} --hyps template --batch_size {bs} --group 200
else:
    print(f"bỏ qua 4b: FINETUNE={FINETUNE}, ROLE={ROLE!r} (cần FINETUNE=True và ROLE là 'all' hoặc 'B')")

## 4c. Đánh giá ngoài miền (chỉ chấm điểm, không huấn luyện)

Với mỗi bộ trong `OOD_SETS`: reader lấy top-5 ứng viên, rồi các verifier trong `OOD_VERIFIERS` chấm với giả thuyết
template. Ngưỡng, λ và bộ kết hợp C sẽ học trên calib của SQuAD (`scripts/ood_eval.py`). Nên chạy cell này **trước**
bước 4b nếu quota GPU có hạn. Ước tính trên T4 (chưa đo): khoảng 2–3 giờ cho cả 4 bộ; chạy dở thì tiếp tục từ chỗ dừng.

In [ ]:
for run_name, (dataset, n) in OOD_SETS.items():
    print(f"===== {run_name}")
    !python scripts/run_reader.py --dataset {dataset} --n {n} --out runs/{run_name} --batch_size 32
    for v in OOD_VERIFIERS:
        ver = v if v in ("nli-xs", "nli-base", "nli-large") else f"runs/models/{v}"
        bs = 16 if "large" in v else 64
        !python scripts/run_verify.py --run runs/{run_name} --verifier {ver} --hyps template --batch_size {bs} --group 200

## 5. Kiểm tra đã đủ điểm của cả 3 verifier chưa

Phần tổng hợp chỉ chạy khi đủ, để không vô tình báo cáo trên một phần dữ liệu.

In [ ]:
def n_ids(path):
    if not os.path.exists(path):
        return 0
    ids = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                ids.add(json.loads(line)["id"])
            except ValueError:  # dòng cuối bị cắt dở khi Colab ngắt; run_verify sẽ tự sửa
                pass
    return len(ids)


status = {v: n_ids(f"runs/{RUN}/scores/{v}.jsonl") for v in ALL_VERIFIERS}
# Tổng hợp cần đủ 3 verifier gốc; mô hình tinh chỉnh nào chưa xong thì analyze.py tự bỏ qua
READY = all(status[v] == N for v in ("nli-xs", "nli-base", "nli-large"))
for v, n in status.items():
    print(f"{v:10s} {n:6d}/{N}", "xong" if n == N else "CHƯA XONG")
print("-> chạy tổng hợp" if READY else "-> chưa đủ: bỏ qua phần tổng hợp, Run all lại khi verifier còn lại xong")

## 6. Bước 3: tổng hợp kết quả

In [ ]:
from IPython.display import Image, Markdown, display
res = f"runs/{RUN}/results"
if READY:
    !python scripts/analyze.py --run runs/{RUN} > /dev/null
    for name in ("main", "significance", "breakdown", "ablation"):
        display(Markdown(open(f"{res}/{name}.md").read()))
    display(Image(f"{res}/risk_coverage.png", width=650))
    display(Image(f"{res}/aurc_vs_size.png", width=500))

## 7. Phân tích bộ dữ liệu và ví dụ cho slide (chỉ cần CPU)

In [ ]:
if READY:
    !python scripts/data_analysis.py --run runs/{RUN} > /dev/null
    display(Markdown(open(f"{res}/data_analysis.md").read()))
    for name in ("data_qtype", "data_overlap", "data_entail", "data_topk"):
        display(Image(f"{res}/{name}.png", width=550))
    !python scripts/deep_analysis.py --run runs/{RUN} > /dev/null
    display(Markdown(open(f"{res}/deep_analysis.md").read()))
    for name in ("deep_overlap_entail", "deep_reliability", "deep_delta_risk", "deep_articles"):
        display(Image(f"{res}/{name}.png", width=550))
    !python scripts/compare_verifiers.py --run runs/{RUN} > /dev/null
    display(Markdown(open(f"{res}/compare_verifiers.md").read()))
    display(Image(f"{res}/compare_hans.png", width=600))
    !python scripts/edge_cases.py --run runs/{RUN} > /dev/null
    !python scripts/error_analysis.py --run runs/{RUN} > /dev/null
    for name in ("edge_cases", "error_analysis"):
        display(Markdown(open(f"{res}/{name}.md").read()))
        display(Image(f"{res}/{name}.png", width=600))
    targets = ",".join(f"runs/{r}" for r in OOD_SETS if os.path.exists(f"runs/{r}/preds.jsonl"))
    if targets:
        !python scripts/ood_eval.py --source runs/{RUN} --targets {targets} > /dev/null
        display(Markdown(open(f"{res}/ood_eval.md").read()))
        display(Image(f"{res}/ood_eval.png", width=600))
        if os.path.exists("runs/addsent/preds.jsonl"):
            !python scripts/ood_examples.py --run runs/addsent --out {res}/ood_examples.md > /dev/null
    !python scripts/risk_control.py --run runs/{RUN} --targets "{targets}" > /dev/null
    display(Markdown(open(f"{res}/risk_control.md").read()))
    display(Image(f"{res}/risk_control.png", width=550))
    !python scripts/find_examples.py --run runs/{RUN} > /dev/null
    print("ví dụ cho slide:", f"{res}/examples.md")

## 8. Gán nhãn câu không đáp án

Ba bộ (xem `annotation/guide.md`): `sheet` (mẫu ngẫu nhiên), `check` (cả hai người gán riêng cùng các câu, để tính kappa),
`errors` (câu C vẫn trả lời). Cell dưới chép `annotation/` từ `squad2k` sang nếu bản 11k chưa có, rồi phân tích bộ nào đã có nhãn.

In [ ]:
import csv
src, dst = "runs/squad2k/annotation", f"runs/{RUN}/annotation"
if RUN != "squad2k" and os.path.isdir(src) and not os.path.isdir(dst):
    !cp -r "{src}" "{dst}"
for name in ("sheet", "check", "errors"):  # mẫu ngẫu nhiên, bộ kiểm tra đồng thuận, lỗi của hệ thống
    sheets = [f"{dst}/{name}_{who}.csv" for who in "AB"]
    labelled = all(os.path.exists(s) for s in sheets) and any(
        r["label"].strip() for s in sheets for r in csv.DictReader(open(s, encoding="utf-8-sig")))
    if READY and labelled:
        !python scripts/analyze_annotation.py --run runs/{RUN} --set {name} > /dev/null
        display(Markdown(open(f"{dst}/{name}_annotation.md").read()))
    else:
        print(f"{name}: chưa có nhãn hoặc chưa đủ verifier, bỏ qua")

## 9. Tải kết quả về

In [ ]:
if READY:
    !cd runs/{RUN} && zip -qr /content/results_{RUN}.zip results
    from google.colab import files
    files.download(f"/content/results_{RUN}.zip")